In [3]:

# Install required libraries
%pip install qiskit qiskit-aer

# Import libraries
from collections import deque
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator

# Step 1: Define initial and goal states
# 0 represents the blank tile

initial_state = (2, 5, 3,
                 1, 8, 6,
                 7, 4, 0)

goal_state = (1, 2, 3,
              4, 5, 6,
              7, 8, 0)

# Step 2: Define possible movements of the blank tile

moves = {
    0: [1, 3],
    1: [0, 2, 4],
    2: [1, 5],
    3: [0, 4, 6],
    4: [1, 3, 5, 7],
    5: [2, 4, 8],
    6: [3, 7],
    7: [4, 6, 8],
    8: [5, 7]
}

# Step 3: Find solution using production rules (BFS)

def solve_puzzle(start, goal):

    queue = deque([(start, [])])
    visited = {start}

    while queue:

        state, path = queue.popleft()

        if state == goal:
            return path

        blank = state.index(0)

        for next_position in moves[blank]:

            new_state = list(state)

            # Apply production rule
            new_state[blank], new_state[next_position] = (
                new_state[next_position],
                new_state[blank]
            )

            new_state = tuple(new_state)

            if new_state not in visited:
                visited.add(new_state)
                queue.append(
                    (new_state, path + [(blank, next_position)])
                )

    return None


# Step 4: Find the sequence of moves

solution_path = solve_puzzle(initial_state, goal_state)

if solution_path is None:

    print("No solution exists for this puzzle.")

else:

    print("Solution found!")
    print("Total moves:", len(solution_path))
    print("Move sequence:", solution_path)

    # Step 5: Create a 36-qubit quantum circuit
    # Each cell uses 4 qubits

    qc = QuantumCircuit(36, 36)

    # Encode initial puzzle state
    for cell, tile in enumerate(initial_state):

        for bit in range(4):

            if (tile >> bit) & 1:
                qc.x(4 * cell + bit)

    qc.barrier()

    # Step 6: Apply quantum SWAP gates

    for blank, next_position in solution_path:

        for bit in range(4):

            qc.swap(
                4 * blank + bit,
                4 * next_position + bit
            )

    qc.barrier()

    # Step 7: Measure all qubits

    qc.measure(range(36), range(36))

    # Step 8: Execute using Matrix Product State simulator

    simulator = AerSimulator(method='matrix_product_state')

    # Use standard gates without backend coupling-map restrictions
    compiled_circuit = transpile(
        qc,
        basis_gates=['u', 'cx'],
        optimization_level=0
    )

    result = simulator.run(
        compiled_circuit,
        shots=1
    ).result()

    counts = result.get_counts()

    # Step 9: Decode measured state

    measured_bits = list(counts.keys())[0].replace(" ", "")[::-1]

    final_state = []

    for cell in range(9):

        value = 0

        for bit in range(4):

            if measured_bits[4 * cell + bit] == '1':
                value |= (1 << bit)

        final_state.append(value)

    # Step 10: Display final puzzle state

    print("\nFinal Puzzle State:")

    for i in range(0, 9, 3):
        print(final_state[i:i + 3])

    print("\nGoal Reached:", tuple(final_state) == goal_state)

    # Display quantum circuit

    print("\nQuantum Circuit:")
    print(qc.draw(fold=120))


Note: you may need to restart the kernel to use updated packages.
Solution found!
Total moves: 16
Move sequence: [(8, 7), (7, 4), (4, 1), (1, 0), (0, 3), (3, 6), (6, 7), (7, 8), (8, 5), (5, 4), (4, 7), (7, 6), (6, 3), (3, 4), (4, 5), (5, 8)]

Final Puzzle State:
[1, 2, 3]
[4, 5, 6]
[7, 8, 0]

Goal Reached: True

Quantum Circuit:
            ░                                                                                                          »
 q_0: ──────░───────────X───────────X──────────────────────────────────────────────────────────────────────────────────»
      ┌───┐ ░           │           │                                                                                  »
 q_1: ┤ X ├─░───────────┼──X────────┼──X───────────────────────────────────────────────────────────────────────────────»
      └───┘ ░           │  │        │  │                                                                               »
 q_2: ──────░───────────┼──┼──X─────┼──┼──X─────────────────────


# Practical 3: Oral Questions and Answers

**Practical Name:** Implement Tarrataca's Quantum Production System with the 3-Puzzle Problem.

## 1. Basic Questions

**Q1. What is the aim of this practical?**

To implement Tarrataca's quantum production system for solving the puzzle problem using Python and Qiskit.

**Q2. What is the 8-puzzle problem?**

It is a puzzle containing 8 numbered tiles and one blank space arranged in a 3×3 grid. The aim is to reach the goal arrangement.

**Q3. What is a quantum production system?**

It represents a problem using states and rules, with quantum operations used to represent or explore possible solutions.

**Q4. What is a puzzle state?**

A state represents the current arrangement of all tiles in the puzzle.

**Q5. What is the initial state?**

The starting arrangement of the puzzle.

**Q6. What is the goal state?**

The required final arrangement of the puzzle.

**Q7. What is a production rule?**

A rule that changes the current state into another valid state.

**Q8. What is the blank tile?**

The empty position in the puzzle, represented by 0 in our program.

**Q9. Why do we use 0 for the blank tile?**

It makes the blank position easy to identify and move using Python.

**Q10. Which software is used in this practical?**

Python, Qiskit and Qiskit Aer.

## 2. Python and BFS

**Q11. What is BFS?**

BFS stands for Breadth-First Search. It explores states level by level.

**Q12. Why do we use BFS?**

To find the shortest solution in terms of the number of moves when every move has equal cost.

**Q13. Which Python library is used for BFS?**

The deque class from the collections module.

**Q14. What is a deque?**

A double-ended queue that allows efficient insertion and removal from both ends.

**Q15. Why do we use a queue?**

To process puzzle states in the order they are discovered.

**Q16. What is the purpose of the visited set?**

It stores already explored states to avoid visiting the same state repeatedly.

**Q17. What does state.index(0) do?**

It finds the position of the blank tile.

**Q18. Why do we use a tuple for puzzle states?**

Tuples are immutable and can be stored in a Python set.

**Q19. What happens if the puzzle has no solution?**

The function returns None and the program displays "No solution exists for this puzzle."

**Q20. What is the time complexity of BFS?**

O(V + E), where V is the number of states and E is the number of transitions.

## 3. Qiskit and Quantum Circuits

**Q21. What is Qiskit?**

Qiskit is an open-source Python framework for quantum computing.

**Q22. What is a quantum circuit?**

A sequence of quantum gates and measurements applied to qubits.

**Q23. What is a qubit?**

A qubit is the basic unit of quantum information.

**Q24. What is the difference between a bit and a qubit?**

A classical bit stores either 0 or 1. A qubit can exist in a superposition of 0 and 1.

**Q25. What is a quantum gate?**

A quantum gate is an operation that changes the state of one or more qubits.

**Q26. Which quantum gates are used in your code?**

X gates and SWAP gates. The code also uses barrier operations.

**Q27. What is an X gate?**

It flips a qubit from 0 to 1 or from 1 to 0.

**Q28. What is a SWAP gate?**

A SWAP gate exchanges the quantum states of two qubits.

**Q29. Why do we use SWAP gates in this practical?**

To exchange the binary-encoded values of two puzzle cells when the blank tile moves.

**Q30. What is a barrier in Qiskit?**

A barrier separates circuit operations and helps organize the circuit.

## 4. Quantum Encoding and Simulation

**Q31. Why does your circuit use 36 qubits?**

There are 9 puzzle cells, and each cell uses 4 qubits.

36 = 9 × 4.

**Q32. Why do we use 4 qubits for each cell?**

Four qubits can represent values from 0 to 15, which is enough to encode puzzle values from 0 to 8.

**Q33. How is a tile encoded into qubits?**

Each tile value is converted into its 4-bit binary representation.

**Q34. What is the purpose of qc.x()?**

It applies an X gate to a qubit when the corresponding binary bit is 1.

**Q35. What is measurement in quantum computing?**

Measurement reads the quantum state and produces classical bits.

**Q36. What is a classical bit?**

A classical bit stores either 0 or 1.

**Q37. What is AerSimulator?**

It is a simulator provided by Qiskit Aer for executing quantum circuits.

**Q38. Why do we use Matrix Product State simulation?**

It provides an alternative simulation method that can handle certain larger circuits more efficiently than a full statevector simulator.

**Q39. What is transpilation?**

Transpilation converts a quantum circuit into a form suitable for execution by a simulator or quantum backend.

**Q40. What does shots=1 mean?**

It means the quantum circuit is executed once.

## 5. Code-Based Questions

**Q41. What does qc.measure(range(36), range(36)) do?**

It measures all 36 qubits and stores their results in 36 classical bits.

**Q42. What does counts = result.get_counts() return?**

It returns a dictionary containing measured bitstrings and their occurrence counts.

**Q43. Why do we reverse the measured bitstring?**

Qiskit displays measurement results in reverse classical-bit order, so reversing helps decode the cells in their original order.

**Q44. What does measured_bits[4 * cell + bit] represent?**

It accesses one measured binary bit belonging to a particular puzzle cell.

**Q45. How do we convert the measured bits back into a tile value?**

We combine the four bits using bit shifting and bitwise OR operations.

**Q46. What does tuple(final_state) == goal_state check?**

It checks whether the final measured puzzle state matches the required goal state.

**Q47. What is the purpose of transpile()?**

It converts the circuit into supported quantum gates before simulation.

**Q48. What does optimization_level=0 mean?**

It requests no additional optimization during transpilation.

**Q49. What is the purpose of basis_gates=['u', 'cx']?**

It restricts the transpiled circuit to the standard U and CNOT gate basis.

**Q50. What is the final output of your practical?**

The program displays the solution path, total moves, final puzzle arrangement, whether the goal is reached, and the quantum circuit.

## 6. Theory Questions

**Q51. What is Grover's algorithm?**

Grover's algorithm is a quantum search algorithm that amplifies the probability of finding a marked solution.

**Q52. What is amplitude amplification?**

It is a quantum technique that increases the probability of measuring desired states.

**Q53. What is QAOA?**

QAOA stands for Quantum Approximate Optimization Algorithm. It is used to find approximate solutions to optimization problems.

**Q54. What is a Hadamard gate?**

A Hadamard gate creates an equal superposition when applied to the computational basis states 0 or 1.

**Q55. What is a CNOT gate?**

A Controlled-NOT gate flips the target qubit when the control qubit is 1.

**Q56. What is superposition?**

Superposition is the ability of a qubit to exist in a combination of quantum states before measurement.

**Q57. What is the difference between Grover's algorithm and QAOA?**

Grover's algorithm is mainly used for quantum search, whereas QAOA is used for approximate optimization problems.

## 7. Important Practical Questions

**Q58. Is your solution completely quantum?**

No. It is a hybrid approach because BFS performs the puzzle search classically, while Qiskit represents and executes the moves using a quantum circuit.

**Q59. Why did you use BFS instead of QAOA?**

The provided QAOA code in the lab manual has compatibility and implementation problems. BFS provides a reliable way to find the shortest legal path for this demonstration.

**Q60. Does the quantum circuit itself find the solution?**

No. The solution path is found by BFS before the quantum circuit is constructed.

**Q61. Can this code solve every 8-puzzle configuration?**

It can find a solution for any solvable configuration, although some configurations may require considerable computation.

**Q62. What happens if two puzzle tiles are not adjacent?**

The program does not allow their direct exchange. Only legal movements of the blank tile are permitted.

**Q63. Why is the final state important?**

It confirms whether the executed quantum circuit produces the required goal arrangement.

## 8. Quick Revision

| Term | Answer |
|---|---|
| BFS | Breadth-First Search |
| Qubit | Basic unit of quantum information |
| Qiskit | Python framework for quantum computing |
| AerSimulator | Quantum circuit simulator |
| X gate | Flips a qubit |
| SWAP gate | Exchanges two qubit states |
| 36 qubits | 9 cells × 4 qubits |
| Deque | Double-ended queue |
| Visited | Avoids repeated states |
| Grover | Quantum search algorithm |
| QAOA | Quantum optimization algorithm |
| Measurement | Converts quantum information into classical results |
| Transpilation | Converts circuit into supported gates |

**Important Note:** This program is a hybrid BFS and quantum-circuit demonstration. BFS finds the solution, and the quantum circuit represents and executes the moves. It does not implement the exact QAOA or Grover algorithm described in the lab manual.
